In [2]:
import sys
!{sys.executable} -m pip install xgboost imbalanced-learn scikit-learn pandas numpy

In [3]:
# ==========================================
# 1. IMPORTS & DATA LOADING
# ==========================================
import pandas as pd
import json
import ast
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import classification_report, accuracy_score
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from xgboost import XGBClassifier

def load_jsonl(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

# Load your specific dataset files
df_train = load_jsonl('train.jsonl')
df_test = load_jsonl('test.jsonl')
df = pd.concat([df_train, df_test], ignore_index=True)


In [4]:
# ==========================================
# 2. FEATURE EXTRACTION & PREPROCESSING
# ==========================================
def extract_label(entities):
    if not entities: return "benign"
    if isinstance(entities, str): entities = ast.literal_eval(entities)
    return entities[0].get('label', 'benign')

df['Threat Category'] = df['entities'].apply(extract_label)
df['text_data'] = df['text'].fillna('no description')

# Custom Tokenizer: Keeps technical markers like C:\, SHA256, and IPs intact
tfidf = TfidfVectorizer(
    max_features=5000, 
    ngram_range=(1, 3), 
    token_pattern=r'[a-zA-Z0-9\\/:\._-]+' 
)

X = tfidf.fit_transform(df['text_data'])
le = LabelEncoder()
y = le.fit_transform(df['Threat Category'])


In [8]:
# ==========================================
# 3. BALANCING THE DATASET
# ==========================================
from collections import Counter

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Before:", sorted(Counter(y_train).items()))

# Only oversample the very small classes (< 50 samples) up to 50
# Leave everything else untouched
class_counts = Counter(y_train)
over_strategy = {cls: max(count, 50) for cls, count in class_counts.items()}

over = SMOTE(sampling_strategy=over_strategy, k_neighbors=1, random_state=42)
X_res, y_res = over.fit_resample(X_train, y_train)

# NO undersampling — keep the majority class as-is
print("After:", sorted(Counter(y_res).items()))


Before: [(np.int64(0), 17), (np.int64(1), 6), (np.int64(2), 104), (np.int64(3), 21), (np.int64(4), 14), (np.int64(5), 6), (np.int64(6), 6), (np.int64(7), 20), (np.int64(8), 59), (np.int64(9), 412), (np.int64(10), 165), (np.int64(11), 42), (np.int64(12), 401), (np.int64(13), 3356), (np.int64(14), 36), (np.int64(15), 378), (np.int64(16), 485), (np.int64(17), 612), (np.int64(18), 279), (np.int64(19), 126), (np.int64(20), 72)]
After: [(np.int64(0), 50), (np.int64(1), 50), (np.int64(2), 104), (np.int64(3), 50), (np.int64(4), 50), (np.int64(5), 50), (np.int64(6), 50), (np.int64(7), 50), (np.int64(8), 59), (np.int64(9), 412), (np.int64(10), 165), (np.int64(11), 50), (np.int64(12), 401), (np.int64(13), 3356), (np.int64(14), 50), (np.int64(15), 378), (np.int64(16), 485), (np.int64(17), 612), (np.int64(18), 279), (np.int64(19), 126), (np.int64(20), 72)]


In [9]:
# ==========================================
# 4. OPTIMIZED HYBRID ENSEMBLE TRAINING
# ==========================================
print("Training High-Speed Hybrid Model... (Est. 2-4 mins)")

xgb_model = XGBClassifier(
    n_estimators=200, 
    learning_rate=0.1, 
    max_depth=6, 
    tree_method='hist', 
    n_jobs=-1, 
    random_state=42
)

rf_model = RandomForestClassifier(
    n_estimators=200, 
    max_depth=15, 
    class_weight='balanced', 
    n_jobs=-1, 
    random_state=42
)

final_model = VotingClassifier(
    estimators=[('xgb', xgb_model), ('rf', rf_model)], 
    voting='soft', 
    n_jobs=-1
)

final_model.fit(X_res, y_res)


Training High-Speed Hybrid Model... (Est. 2-4 mins)


,estimators,"[('xgb', ...), ('rf', ...)]"
,voting,'soft'
,weights,None
,n_jobs,-1
,flatten_transform,True
,verbose,False
,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None


In [10]:
# ==========================================
# 5. EVALUATION
# ==========================================
y_pred = final_model.predict(X_test)
print(f"\n--- [FINAL RESULTS] ---")
print(f"Overall Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))



--- [FINAL RESULTS] ---
Overall Accuracy: 63.99%
                precision    recall  f1-score   support

        DOMAIN       0.00      0.00      0.00         4
         EMAIL       1.00      0.50      0.67         2
      FILEPATH       0.00      0.00      0.00        26
          IPV4       0.50      0.40      0.44         5
 Infrastucture       0.00      0.00      0.00         3
           MD5       0.00      0.00      0.00         2
   REGISTRYKEY       0.00      0.00      0.00         1
          SHA1       1.00      0.20      0.33         5
          SHA2       0.00      0.00      0.00        15
      SOFTWARE       0.50      0.20      0.29       103
          TIME       0.36      0.12      0.18        41
           URL       0.67      0.20      0.31        10
attack-pattern       0.58      0.38      0.46       101
        benign       0.66      0.97      0.78       840
      campaign       0.75      0.33      0.46         9
      identity       0.59      0.43      0.49        

In [11]:
def defense_portal_triage(report_text):
    text_lower = report_text.lower()
    manual_category = None
    priority_boost = 0 # New variable to force high risk
    
    # 1. ENHANCED RULES (Forensic Keywords)
    if "sha" in text_lower or "hash" in text_lower: 
        manual_category = "SHA2"
    elif "ip" in text_lower or "address" in text_lower: 
        manual_category = "IPV4"
    
    # 2. CRITICAL THREAT BOOST (Force Critical Priority)
    critical_keywords = ["unauthorized", "wannacry", "malware", "attack", "exploit"]
    if any(word in text_lower for word in critical_keywords):
        priority_boost = 8.0 # Minimum score for critical threats
        if "unauthorized" in text_lower: manual_category = "ATTACK-PATTERN"
        if "wannacry" in text_lower or "malware" in text_lower: manual_category = "MALWARE"

    # 3. AI CLASSIFICATION
    vec = tfidf.transform([report_text])
    probs = final_model.predict_proba(vec)
    ai_confidence = np.max(probs)
    ai_category = le.classes_[np.argmax(probs)]
    
    # 4. FINAL LOGIC
    final_category = manual_category if manual_category else ai_category
    
    # Final Risk = (AI Confidence * 10) + Priority Boost (Capped at 10)
    risk_score = min(10.0, (ai_confidence * 10) + priority_boost)
    
    print(f"\n--- [CERT-DEFENSE FINAL TRIAGE] ---")
    print(f"Final Classification: {final_category.upper()}")
    print(f"Calculated Risk: {round(risk_score, 1)}/10")
    print(f"Priority Level: {'CRITICAL' if risk_score >= 7.5 else 'HIGH' if risk_score >= 5.0 else 'MEDIUM'}")

In [12]:
# ==========================================
# 7. EXECUTION (Run this to see the output)
# ==========================================
scenarios = [
    "Suspicious SHA256 hash detected in system memory during routine sweep.",
    "Unauthorized access attempt detected from IP 192.168.1.50 targeting the personnel database.",
    "System log: Routine backup completed successfully for server cluster 4.",
    "Known malware signature 'WannaCry' identified in encrypted network traffic."
]

print("--- STARTING LIVE DEFENSE PORTAL TRIAGE ---")
for incident in scenarios:
    defense_portal_triage(incident)

--- STARTING LIVE DEFENSE PORTAL TRIAGE ---

--- [CERT-DEFENSE FINAL TRIAGE] ---
Final Classification: SHA2
Calculated Risk: 3.9/10
Priority Level: MEDIUM

--- [CERT-DEFENSE FINAL TRIAGE] ---
Final Classification: ATTACK-PATTERN
Calculated Risk: 10.0/10
Priority Level: CRITICAL

--- [CERT-DEFENSE FINAL TRIAGE] ---
Final Classification: BENIGN
Calculated Risk: 4.0/10
Priority Level: MEDIUM

--- [CERT-DEFENSE FINAL TRIAGE] ---
Final Classification: MALWARE
Calculated Risk: 10.0/10
Priority Level: CRITICAL
